In [ ]:
import sys, json, random
from pathlib import Path

import numpy as np
import torch

repo = Path("..").resolve()
sys.path.insert(0, str(repo))

from src.memory import PrototypeMemory


In [ ]:
FEATURES_DIR = repo / "outputs/notebook/encode/features"   
MEMORY_PATH  = repo / "outputs/notebook/memory/memory.pt"  
OUT_PATH     = FEATURES_DIR / "predictions.json"           

DEVICE = "cuda"


In [ ]:
features = np.load(FEATURES_DIR / "features.npy")
records  = json.loads((FEATURES_DIR / "records.json").read_text())
memory   = PrototypeMemory.load(MEMORY_PATH, device=DEVICE)

preds = []
for r in records:
    idx = int(r["feature_index"])
    if idx < 0 or idx >= features.shape[0]:
        continue
    z = torch.from_numpy(features[idx]).float().to(memory.device)
    z = memory.prepare_for_inference(z)
    pred, conf = memory.predict(z, prepared=True)
    preds.append({
        "image_path":    r.get("image_path"),
        "box_xyxy":      r.get("box_xyxy"),
        "feature_index": idx,
        "score":         float(r.get("score", 0.0)),
        "gt_category":   r.get("gt_category"),
        "pred_label":    pred,
        "pred_conf":     float(conf),
    })

OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
OUT_PATH.write_text(json.dumps(preds, indent=2))
print(f"features : {features.shape}")
print(f"records  : {len(records)}")
print(f"predicted: {len(preds)}")
print(f"saved  -> {OUT_PATH}")


In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

N        = 10          # how many crops to show
MIN_CONF = 0.0         # set e.g. 0.3 to show only confident predictions

sample = [p for p in preds if p["pred_conf"] >= MIN_CONF]
sample = random.sample(sample, min(N, len(sample)))

cols = min(5, len(sample))
rows = (len(sample) + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(3 * cols, 3 * rows))
axes = np.array(axes).reshape(-1)

for ax, p in zip(axes, sample):
    img = np.asarray(Image.open(p["image_path"]).convert("RGB"))
    x1, y1, x2, y2 = [int(round(v)) for v in p["box_xyxy"]]
    crop = img[y1:y2, x1:x2]
    ax.imshow(crop)
    gt  = p.get("gt_category") or "?"
    ax.set_title(
        f"pred: {p['pred_label']}\nconf: {p['pred_conf']:.2f}  gt: {gt}",
        fontsize=7,
        color="green" if p["pred_label"] == gt else "red",
    )
    ax.axis("off")

for ax in axes[len(sample):]:
    ax.axis("off")

fig.suptitle("Prediction sanity check", fontsize=11)
plt.tight_layout()
plt.show()